We will analyze the Porto Seguro's Safe Driver dataset. The goal is to predict whether a driver will make an insurance claim next year.

The evaulation metric: Normalized Gini Coefficient. The code for comupting the metric was taken from the Kaggle competition site.

The dataset contains 595,212 samples over 57 features (all numeric) without missing values. It has a binary traget value of whether an insurance was claimed or not. Since the dataset size is about 34M entries, dimensionally reduction is a crucial step to handle the CPU/Memory load.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import make_scorer
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.utils import class_weight

In [2]:
def gini(actual, pred, cmpcol = 0, sortcol = 1):
    assert( len(actual) == len(pred) )
    all = np.asarray(np.c_[actual, pred, np.arange(len(actual))], dtype=float)
    all = all[np.lexsort((all[:,2], -1*all[:,1]))]
    totalLosses = all[:,0].sum()
    giniSum = all[:,0].cumsum().sum() / totalLosses

    giniSum -= (len(actual) + 1) / 2.
    return giniSum / len(actual)

def gini_normalized(a, p):
    return gini(a, p) / gini(a, a)

In [3]:
df = pd.read_csv('train.csv')
print("First 5 rows of train:")
print(df.head())

First 5 rows of train:
   id  target  ps_ind_01  ps_ind_02_cat  ps_ind_03  ps_ind_04_cat  \
0   7       0          2              2          5              1   
1   9       0          1              1          7              0   
2  13       0          5              4          9              1   
3  16       0          0              1          2              0   
4  17       0          0              2          0              1   

   ps_ind_05_cat  ps_ind_06_bin  ps_ind_07_bin  ps_ind_08_bin  ...  \
0              0              0              1              0  ...   
1              0              0              0              1  ...   
2              0              0              0              1  ...   
3              0              1              0              0  ...   
4              0              1              0              0  ...   

   ps_calc_11  ps_calc_12  ps_calc_13  ps_calc_14  ps_calc_15_bin  \
0           9           1           5           8               0   
1  

In [4]:
print(df.shape)
print(df.info())
print(df.describe())
print(df['target'].value_counts())


(595212, 59)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 595212 entries, 0 to 595211
Data columns (total 59 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   id              595212 non-null  int64  
 1   target          595212 non-null  int64  
 2   ps_ind_01       595212 non-null  int64  
 3   ps_ind_02_cat   595212 non-null  int64  
 4   ps_ind_03       595212 non-null  int64  
 5   ps_ind_04_cat   595212 non-null  int64  
 6   ps_ind_05_cat   595212 non-null  int64  
 7   ps_ind_06_bin   595212 non-null  int64  
 8   ps_ind_07_bin   595212 non-null  int64  
 9   ps_ind_08_bin   595212 non-null  int64  
 10  ps_ind_09_bin   595212 non-null  int64  
 11  ps_ind_10_bin   595212 non-null  int64  
 12  ps_ind_11_bin   595212 non-null  int64  
 13  ps_ind_12_bin   595212 non-null  int64  
 14  ps_ind_13_bin   595212 non-null  int64  
 15  ps_ind_14       595212 non-null  int64  
 16  ps_ind_15       595212 non-null  int64  
 1

21,694/573,518 drivers claimed insurance suggesting imbalanced dataset.

## Data Preprocessing for Modeling

In [5]:
X = df.drop(['id', 'target'], axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Shape of X_train_scaled: {X_train_scaled.shape}")
print(f"Shape of X_test_scaled: {X_test_scaled.shape}")

Shape of X_train_scaled: (476169, 57)
Shape of X_test_scaled: (119043, 57)


## Dimensionality Reduction with PCA and Nyström Kernel PCA

In [6]:
n_components=20

print("\nApplying PCA")
pca = PCA(n_components=n_components, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"Shape after PCA: {X_train_pca.shape}")

log_reg_pca = LogisticRegression(random_state=42, solver='liblinear', class_weight='balanced')
log_reg_pca.fit(X_train_pca, y_train)
y_pred_pca = log_reg_pca.predict(X_test_pca)
y_proba_pca = log_reg_pca.predict_proba(X_test_pca)[:, 1]

gini_pca = gini_normalized(y_test, y_proba_pca)

print(f"PCA + Logistic Regression Normalized Gini: {gini_pca:.4f}")


Applying PCA
Shape after PCA: (476169, 20)
PCA + Logistic Regression Normalized Gini: 0.2353


In [7]:
print("\nApplying Nyström Kernel PCA")
nystroem = Nystroem(kernel='rbf', n_components=n_components, random_state=42)
X_train_nystroem = nystroem.fit_transform(X_train_scaled)
X_test_nystroem = nystroem.transform(X_test_scaled)

print(f"Shape after Nyström Kernel PCA: {X_train_nystroem.shape}")

log_reg_nystroem = LogisticRegression(random_state=42, solver='liblinear', class_weight='balanced')
log_reg_nystroem.fit(X_train_nystroem, y_train)
y_pred_nystroem = log_reg_nystroem.predict(X_test_nystroem)
y_proba_nystroem = log_reg_nystroem.predict_proba(X_test_nystroem)[:, 1]

gini_nystroem = gini_normalized(y_test, y_proba_nystroem)

print(f"Nyström Kernel PCA + Logistic Regression Normalized Gini: {gini_nystroem:.4f}")


Applying Nyström Kernel PCA
Shape after Nyström Kernel PCA: (476169, 20)
Nyström Kernel PCA + Logistic Regression Normalized Gini: 0.2225


It looks like that the data is already linear since the kernal PCA performance was a bit worse than the regular PCA. Will continue with the regular PCA.

### 5-Fold Cross-Validation Model Comparison

In [8]:
n_components = 0.8
pca = PCA(n_components=n_components, random_state=42)
X_scaled = scaler.fit_transform(X)
X_pca = pca.fit_transform(X_scaled)
print(f"Shape after PCA: {X_pca.shape}")

Shape after PCA: (595212, 36)


In [9]:
def run_cv(model, X, y, model_name, n_splits=5, early_stopping_rounds=None):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    gini_scores = []

    print(f"Running {n_splits}-Fold Cross-Validation for {model_name}:")
    for fold, (train_index, test_index) in enumerate(skf.split(X, y)):
        if isinstance(X, pd.DataFrame):
            X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
        else:
            X_train_fold, X_test_fold = X[train_index], X[test_index]
        y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

        fit_params = {}
        if isinstance(model, lgb.LGBMClassifier) and early_stopping_rounds is not None and early_stopping_rounds > 0:
            fit_params['eval_set'] = [(X_test_fold, y_test_fold)]
            fit_params['eval_metric'] = 'logloss'
            fit_params['callbacks'] = [lgb.early_stopping(stopping_rounds=early_stopping_rounds, verbose=False)]

        model.fit(X_train_fold, y_train_fold, **fit_params)

        if isinstance(model, lgb.LGBMClassifier) and 'callbacks' in fit_params and hasattr(model, 'best_iteration_') and model.best_iteration_ is not None:
            y_proba_fold = model.predict_proba(X_test_fold, num_iteration=model.best_iteration_)[:, 1]
        else:
            y_proba_fold = model.predict_proba(X_test_fold)[:, 1]

        gini = gini_normalized(y_test_fold, y_proba_fold)
        gini_scores.append(gini)
        print(f"Fold {fold+1} Gini: {gini:.4f}")

    avg_gini = np.mean(gini_scores)
    std_gini = np.std(gini_scores)
    print(f"Average Gini for {model_name}: {avg_gini:.4f} (+/- {std_gini:.4f})")
    return avg_gini, std_gini

In [20]:
log_reg_cv = LogisticRegression(random_state=42, solver='liblinear', class_weight='balanced')
avg_gini_log_reg, std_gini_log_reg = run_cv(log_reg_cv, X_pca, y, 'logistic_reg')


Running 5-Fold Cross-Validation for logistic_reg:
Fold 1 Gini: 0.2389
Fold 2 Gini: 0.2296
Fold 3 Gini: 0.2325
Fold 4 Gini: 0.2378
Fold 5 Gini: 0.2246
Average Gini for logistic_reg: 0.2327 (+/- 0.0053)


In [11]:
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'learning_rate': 0.01,
    'n_estimators': 100
}

In [12]:
xgb_cv = XGBClassifier(**params, random_state=42)
avg_gini_xgb, std_gini_xgb = run_cv(xgb_cv, X, y, 'XGBoost')


Running 5-Fold Cross-Validation for XGBoost:
Fold 1 Gini: 0.2700
Fold 2 Gini: 0.2554
Fold 3 Gini: 0.2605
Fold 4 Gini: 0.2599
Fold 5 Gini: 0.2475
Average Gini for XGBoost: 0.2586 (+/- 0.0073)


In [13]:
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'num_leaves': 31,
    'min_data_in_leaf': 1500,
    'learning_rate': 0.01,
    'n_estimators': 2000,
    'verbose': 0
}

In [14]:
lgbm_cv = lgb.LGBMClassifier(**params, random_state=42)
avg_gini_lgbm, std_gini_lgbm = run_cv(lgbm_cv, X, y, 'LightGBM', early_stopping_rounds=1400)

Running 5-Fold Cross-Validation for LightGBM:
[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500
[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500
[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500
[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500
Fold 1 Gini: 0.2906
[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500
[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500
[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500
[LightGBM] [Warning] min_data_in_leaf is set=1500, m

### Cross-Validation Summary

In [15]:
print("5-Fold Cross-Validation Results")
print(f"Logistic Regression (PCA) Average Gini = {avg_gini_log_reg:.4f} (+/- {std_gini_log_reg:.4f})")
print(f"XGBoost (No PCA) Average Gini = {avg_gini_xgb:.4f} (+/- {std_gini_xgb:.4f})")
print(f"LightGBM (No PCA) Average Gini = {avg_gini_lgbm:.4f} (+/- {std_gini_lgbm:.4f})")


5-Fold Cross-Validation Results
Logistic Regression (PCA) Average Gini = 0.2327 (+/- 0.0053)
XGBoost (No PCA) Average Gini = 0.2586 (+/- 0.0073)
LightGBM (No PCA) Average Gini = 0.2821 (+/- 0.0067)


The best model by 5-fold CV is LightGBM. We will report the results based on this model.

In [16]:
test = pd.read_csv('test.csv')
test_id = test['id']
test = test.drop('id', axis=1)

In [17]:
lgbm = lgb.LGBMClassifier(**params, random_state=42)
lgbm.fit(X, y)

if hasattr(lgbm, 'best_iteration_') and lgbm.best_iteration_ is not None:
    y_proba_test = lgbm.predict_proba(test, num_iteration=lgbm.best_iteration_)[:, 1]
else:
    y_proba_test = lgbm.predict_proba(test)[:, 1]



[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500
[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500
[LightGBM] [Warning] min_data_in_leaf is set=1500, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=1500


In [18]:
submission = pd.DataFrame({'id': test_id, 'target': y_proba_test})

submission = submission.sort_values(by='target', ascending=False).reset_index(drop=True)

submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully.")
print(submission.head())

Submission file 'submission.csv' created successfully.
        id    target
0   973312  0.442342
1  1137681  0.391070
2   403927  0.384405
3  1204345  0.376785
4    76859  0.371628


Final test score on submission: 0.24118.
Obviously, If we used all available features without PCA, the results could have been much better. Without PCA, lightGBM resulted with average CV 0.2821, and kaggle private:public scores 0.28005:0.27267.